In [1]:
import pandas as pd
import numpy as np


# Анализ полученных данных. 

* по полученным ранее данным, считаем что сплит является репрезентабельным:
   - разделение пользователей было случайным;
   - разбиение детерменированное (функция разделеня пользователей было создана и протестирована во время проведения А/А теста)
   - выборки независимы (нет пользователей, попавших в обе группы одновременно)

План действий:
 - Преобразование данных:
        - загрузка из внешнего источтика (Excel-файл)
        - проверка корректности загрузки и типов данных
        - отсутствие пустых или null значений (если присутствуют, то удаляем данные, проверяем на репрезентативность) 
 - Анализ распределения по группам, платформам, издателям
 - Сохранение очищенного датафрема в csv-файл
       

In [2]:
# импортируем отчет из xlsx-файла в DataFrame
df = pd.read_excel('../data/raw/marketpele_ab_test.xlsx')

cnt_rows = len(df) # текущий размер данных
print(f'Отчет содержит {cnt_rows} строк')
# проверка на отсутствие пустых или null значений (если присутствуют, то удаляем данные) 
exist_null = df.isna().any().any()
exist_dublecat = df.duplicated().sum() 
if exist_null or exist_dublecat:
    df = df.dropna()
    df = df.drop_duplicates()
    print(f'В данных присутствуют пустые/дублирующие значения. Данные строки будут удалены')
    cnt_rows_notnull = len(df)
    print(f'''Было удалено {cnt_rows - cnt_rows_notnull} строк. 
          Проверим сохранение пропорциональности выборок и репризентабильность (количество наблюдений не меньше 600 тыс. для каждой группы)'''
          )
else:
    print('Данне не содержат пустых значений и дубликатов. Можно проводить дальнейший анализ')   


Отчет содержит 126 строк
Данне не содержат пустых значений и дубликатов. Можно проводить дальнейший анализ


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               126 non-null    datetime64[ns]
 1   publisher_id       126 non-null    int64         
 2   platform           126 non-null    object        
 3   group_name         126 non-null    object        
 4   pageviews          126 non-null    int64         
 5   visible_pageviews  126 non-null    int64         
 6   sessions           126 non-null    int64         
 7   revenue            126 non-null    float64       
 8   sponsord_clicks    126 non-null    int64         
 9   organic_clicks     126 non-null    int64         
dtypes: datetime64[ns](1), float64(1), int64(6), object(2)
memory usage: 10.0+ KB


In [10]:
df.head()

,date,publisher_id,platform,group_name,pageviews,visible_pageviews,sessions,revenue,sponsord_clicks,organic_clicks
0,2019-03-01,101,Desktop,A,16580,5418,12684,34.5201,300,849
1,2019-03-01,101,Desktop,B,16191,4906,12717,32.9211,268,555
2,2019-03-01,106,Desktop,A,16227,11395,11750,20.1620,601,2425
3,2019-03-01,106,Desktop,B,15060,10369,11967,19.7429,663,1194
4,2019-03-01,106,Mobile,A,30494,7715,22843,82.9329,456,1600


In [11]:
general_info = df.nunique() 
print(f'''
Данные за {general_info.date} дней.
Количество групп: {general_info.group_name}.
Количество платформ: {general_info.platform}.
Количество издательств: {general_info.publisher_id}.
''')


Данные за 7 дней.
Количество групп: 2.
Количество платформ: 2.
Количество издательств: 8.



In [12]:
# проверка сплита 
# Категориальные метрики: платформа (platform), издмтельство (publisher_id)
# присутствие всех категориальных значений  в каждой группе
split1 = df.groupby('group_name').agg(count_platform = ('platform', 'nunique'), 
                                            name_platform = ('platform', frozenset),
                                            count_publusher =  ('publisher_id','nunique'),
                                            name_publusher =  ('publisher_id',frozenset)
                                            )
if split1.duplicated(keep=False).sum() == len(split1):
    print ('Сплитование групп корректно. Группы (контрольна А, тестовая В) содержат все платформы и издательства')
else:
    print ('Сплитование не корректно, так как группы содержат разных участников эксперимента')

Сплитование групп корректно. Группы (контрольна А, тестовая В) содержат все платформы и издательства


In [13]:
# Издатели могут быть представлены на разных платформах. Проверим как они представлены на них
df.groupby('publisher_id').agg(name_platform = ('platform',frozenset),
                                       count_platform = ('platform', 'nunique')
)

# Это важно: всего один издатель представлен на двух платформах, остальные только по одной! Это повлияет на анализ А/В теста.
# Необходимо будет его проводить в разрезе платформ. 

,name_platform,count_platform
publisher_id,,
101,(Desktop),1
106,"(Mobile, Desktop)",2
111,(Desktop),1
123,(Desktop),1
259,(Mobile),1
373,(Desktop),1
574,(Mobile),1
700,(Mobile),1


In [4]:
# выгрузим данные в csv файл для дальнейшего анализа
df.to_csv('../data/raw/data.csv', index=False)